# ch04 Bonus 07：差分自注意力（Differential Self-Attention, DSA）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/09_dsa
> **参考真实模型**：Sakana AI 的 DSA Transformer（2025）

## 一句话

用**两个注意力分支相减**，靠差分抵消 softmax 注意力的噪声，提升信噪比。出自 Sakana AI 2025 的研究。

## 动机：softmax 注意力的噪声

标准 softmax 对所有 key 都分配正的注意力权重（即使不相干）。这些本应接近 0 的小权重会累积成噪声，干扰信号。实验观察到注意力图常呈现'双峰'：一个信号峰 + 一个噪声峰。

DSA 的洞察：**两个注意力图相减，让噪声峰相互抵消，信号峰被强化**。

## 核心公式

标准注意力：$Y = \text{softmax}(QK^T) V$

差分注意力：$Y = (\text{softmax}(Q_1 K^T) - \lambda \cdot \text{softmax}(Q_2 K^T)) V$

- 用两套 query（`Q1`、`Q2`）算两份注意力图
- `λ` 是可学习的差分系数，初始化使两个分支在训练初期接近抵消
- 相减后，共同噪声被消去，留下真正的注意力信号

> 论文报告：DSA 在相同参数量下比标准注意力提升约 5%-10% 的 zero-shot 性能。

In [ ]:
import torch
import torch.nn as nn


class DifferentialSelfAttention(nn.Module):
    """差分自注意力：两个注意力分支相减，抑制噪声。"""

    def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        # 两套 query（这是 DSA 与标准注意力的关键区别）
        self.W_q1 = nn.Linear(d_in, d_out, bias=False)
        self.W_q2 = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)
        # 可学习的 λ 参数：λ = exp(λq1·λk1) - exp(λq2·λk2) + λ_init
        self.lambda_init = 0.8
        self.lambda_q1 = nn.Parameter(torch.ones(num_heads) * 0.5)
        self.lambda_k1 = nn.Parameter(torch.ones(num_heads) * 0.5)
        self.lambda_q2 = nn.Parameter(torch.ones(num_heads) * 0.5)
        self.lambda_k2 = nn.Parameter(torch.ones(num_heads) * 0.5)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool(),
        )

    def forward(self, x):
        b, n, _ = x.shape
        H, hd = self.num_heads, self.head_dim
        q1 = self.W_q1(x).view(b, n, H, hd).transpose(1, 2)
        q2 = self.W_q2(x).view(b, n, H, hd).transpose(1, 2)
        k  = self.W_k(x).view(b, n, H, hd).transpose(1, 2)
        v  = self.W_v(x).view(b, n, H, hd).transpose(1, 2)

        mask_bool = self.mask.bool()[:n, :n]
        # 两个分支的注意力分数
        s1 = q1 @ k.transpose(2, 3)
        s2 = q2 @ k.transpose(2, 3)
        s1.masked_fill_(mask_bool, -torch.inf)
        s2.masked_fill_(mask_bool, -torch.inf)
        a1 = torch.softmax(s1 / hd ** 0.5, dim=-1)
        a2 = torch.softmax(s2 / hd ** 0.5, dim=-1)

        # 可学习差分系数 λ
        lam = (torch.exp(self.lambda_q1 * self.lambda_k1)
               - torch.exp(self.lambda_q2 * self.lambda_k2)
               + self.lambda_init)
        lam = lam.view(1, H, 1, 1)

        # 核心差分：a1 - λ·a2
        attn = a1 - lam * a2
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(out)

## 2. 运行 DSA，观察差分效果

In [ ]:
torch.manual_seed(123)
batch, seq, dim, n_heads = 2, 8, 768, 12
x = torch.randn(batch, seq, dim)

dsa = DifferentialSelfAttention(dim, dim, 1024, n_heads)
out = dsa(x)
print(f"DSA 输出: {tuple(out.shape)}")

# 观察 λ 的当前值（初始化应使 a1 - λ·a2 在训练初期接近抵消噪声）
with torch.no_grad():
    lam_val = (torch.exp(dsa.lambda_q1 * dsa.lambda_k1)
               - torch.exp(dsa.lambda_q2 * dsa.lambda_k2)
               + dsa.lambda_init)
    print(f"\n各头 λ 初值: {lam_val.tolist()}")
    print(f"λ 均值: {lam_val.mean():.3f}（接近 1 时两分支近似抵消，突出差分信号）")
print(f"\n💡 相比标准注意力，DSA 多了一套 query 和 λ 参数，代价是参数量略增。")

---
> 📌 本 notebook 实现 DSA 差分注意力核心并验证 λ 初始化。
> 完整集成进 Transformer 的实现见官方 `ch04/09_dsa`。